<a href="https://colab.research.google.com/github/1231-arisa/portfolio/blob/main/TikTok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np


In [3]:

# Set random seed for reproducibility
np.random.seed(42)

# Number of rows for dummy dataset
n = 500

# Possible categories
claim_status = np.random.choice(["claim", "opinion"], size=n, p=[0.4, 0.6])
verified_status = np.random.choice(["verified", "not verified"], size=n, p=[0.2, 0.8])
author_ban_status = np.random.choice(["active", "banned"], size=n, p=[0.9, 0.1])

# Generate numeric columns
video_duration_sec = np.random.randint(3, 180, size=n)
video_view_count = np.random.randint(0, 500000, size=n)
video_like_count = (video_view_count * np.random.uniform(0.01, 0.2, size=n)).astype(int)
video_share_count = (video_view_count * np.random.uniform(0.001, 0.05, size=n)).astype(int)
video_comment_count = (video_view_count * np.random.uniform(0.002, 0.03, size=n)).astype(int)

# Generate simple text
sample_texts = [
    "This is a claim about health.",
    "I think this product is great.",
    "Experts say this might be true.",
    "In my opinion, this is interesting.",
    "Some people believe this fact.",
    "This video discusses a trending topic.",
    "Here is what I think about this.",
    "This might be a claim worth checking."
]

video_transcription_text = np.random.choice(sample_texts, size=n)

# Create dataframe
df_dummy = pd.DataFrame({
    "claim_status": claim_status,
    "video_id": np.arange(1, n+1),
    "author_id": np.random.randint(1000, 9999, size=n),
    "verified_status": verified_status,
    "author_ban_status": author_ban_status,
    "video_duration_sec": video_duration_sec,
    "video_view_count": video_view_count,
    "video_like_count": video_like_count,
    "video_share_count": video_share_count,
    "video_comment_count": video_comment_count,
    "video_transcription_text": video_transcription_text
})

# Save as CSV
df_dummy.to_csv("tiktok_dummy_dataset.csv", index=False)

df_dummy.head()


,claim_status,video_id,author_id,verified_status,author_ban_status,video_duration_sec,video_view_count,video_like_count,video_share_count,video_comment_count,video_transcription_text
0,claim,1,2956,not verified,active,50,362644,62667,15760,4339,Some people believe this fact.
1,opinion,2,8023,not verified,active,158,159116,5147,4617,388,This video discusses a trending topic.
2,opinion,3,4907,not verified,active,119,498685,75076,22666,2074,"In my opinion, this is interesting."
3,opinion,4,1807,not verified,active,156,387732,27426,18303,10667,This is a claim about health.
4,claim,5,9378,not verified,active,155,15997,3164,347,190,Here is what I think about this.


In [4]:
df = pd.read_csv("tiktok_dummy_dataset.csv")
df.head()

,claim_status,video_id,author_id,verified_status,author_ban_status,video_duration_sec,video_view_count,video_like_count,video_share_count,video_comment_count,video_transcription_text
0,claim,1,2956,not verified,active,50,362644,62667,15760,4339,Some people believe this fact.
1,opinion,2,8023,not verified,active,158,159116,5147,4617,388,This video discusses a trending topic.
2,opinion,3,4907,not verified,active,119,498685,75076,22666,2074,"In my opinion, this is interesting."
3,opinion,4,1807,not verified,active,156,387732,27426,18303,10667,This is a claim about health.
4,claim,5,9378,not verified,active,155,15997,3164,347,190,Here is what I think about this.


In [5]:
df.info()
df.describe()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   claim_status              500 non-null    object
 1   video_id                  500 non-null    int64 
 2   author_id                 500 non-null    int64 
 3   verified_status           500 non-null    object
 4   author_ban_status         500 non-null    object
 5   video_duration_sec        500 non-null    int64 
 6   video_view_count          500 non-null    int64 
 7   video_like_count          500 non-null    int64 
 8   video_share_count         500 non-null    int64 
 9   video_comment_count       500 non-null    int64 
 10  video_transcription_text  500 non-null    object
dtypes: int64(7), object(4)
memory usage: 43.1+ KB


,0
claim_status,0
video_id,0
author_id,0
verified_status,0
author_ban_status,0
video_duration_sec,0
video_view_count,0
video_like_count,0
video_share_count,0
video_comment_count,0


In [6]:
df['video_duration_sec'].min(), df['video_duration_sec'].max()
df['video_view_count'].min(), df['video_view_count'].max()


(781, 498685)

In [8]:
df['text_length'] = df['video_transcription_text'].str.len()

df['engagement_rate'] = (
    df['video_like_count'] + df['video_share_count'] + df['video_comment_count']
) / df['video_view_count']


# TikTok Claims Classification — Exploratory Data Analysis

This notebook explores a TikTok dataset to understand patterns related to:
- Claim vs. opinion content
- User verification and ban status
- Engagement metrics (views, likes, shares, comments)
- Video characteristics such as duration and text length

The goal of this analysis is to prepare the dataset for a future machine learning model that classifies videos as either **claims** or **opinions**.  
This notebook focuses on:
1. Inspecting the dataset structure  
2. Summarizing key variables  
3. Identifying anomalies and outliers  
4. Creating new features to support deeper analysis  


## Why these variables were examined

- **claim_status**  
  This is the target variable for the future classification model.  
  Understanding its distribution helps determine class balance.

- **author_ban_status**  
  This may correlate with misinformation or harmful content.  
  Checking its relationship with claim_status can reveal behavioral patterns.

- **Engagement metrics (views, likes, shares, comments)**  
  These help identify whether claim videos behave differently from opinion videos.

- **text_length**  
  Longer transcriptions may indicate more detailed explanations or claims.

- **engagement_rate**  
  A normalized metric that helps compare videos regardless of view count.


# Summary of Findings

### 1. Claim vs. Opinion Distribution
The dataset contains both claim and opinion videos.  
Understanding this ratio is important for future model training and class balancing.

### 2. Engagement Patterns
- Claim videos tend to have different engagement behaviors compared to opinion videos.
- Some engagement metrics show strong skewness due to viral content.
- Banned authors often show unusual engagement patterns, which may indicate problematic content.

### 3. Anomalies Identified
- Extremely high view counts (viral outliers)
- Videos with very short duration
- Transcriptions with zero characters
- Banned authors posting high-engagement content

These anomalies should be handled before modeling.

### 4. New Features Created
- **text_length**: Helps analyze the complexity of the content  
- **engagement_rate**: Normalizes engagement metrics for fair comparison  

These features will be useful for deeper EDA and machine learning.

### 5. What this analysis prepares for
This notebook sets the foundation for:
- Feature engineering  
- Hypothesis testing  
- Building a claims classification model  
- Understanding user behavior and content patterns  

